# MegaAI · M1 — `ui-detector` (YOLO)

Trains the model that **looks at a screenshot and finds every UI element** — button, input,
link, checkbox, radio, select, textarea, image, heading, nav — with a bounding box for each.
This is pixel-level vision: it needs no DOM, so it also works on screenshots of native apps.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

**Dataset:** generated by MegaAI itself — synthetic pages rendered in headless Chromium with
boxes read straight from the DOM, so every label is exact and there is no manual annotation.

**Output:** `ui-detector.pt` (PyTorch) and `ui-detector.onnx` (runs in the Node engine).


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — in Colab use Runtime -> Change runtime type -> T4 GPU (training will be slow otherwise).")


In [ ]:
!pip install -q ultralytics onnx onnxruntime


## 1. Load the dataset


In [ ]:
# Upload the dataset.zip produced by:
#   node scripts/dataset/generate-ui-dataset.mjs --count 3000 --out dataset
#   zip -r dataset.zip dataset
import os, zipfile

if not os.path.exists("dataset"):
    try:
        from google.colab import files
        print("Choose your dataset.zip …")
        up = files.upload()
        name = list(up.keys())[0]
    except Exception:
        name = "dataset.zip"          # running locally: put dataset.zip beside the notebook
    with zipfile.ZipFile(name) as z:
        z.extractall(".")

# The zip may contain dataset/ at the root or one level down — find data.yaml.
root = None
for base, dirs, fs in os.walk("."):
    if "data.yaml" in fs and "images" in dirs:
        root = os.path.abspath(base)
        break
assert root, "Could not find the dataset (no data.yaml with an images/ folder)"
print("dataset root:", root)
print("train images:", len(os.listdir(os.path.join(root, "images/train"))))
print("val images:  ", len(os.listdir(os.path.join(root, "images/val"))))


## 2. Sanity-check the labels

Draws boxes on one training image — they should sit exactly on the elements.


In [ ]:
import os, random
from PIL import Image, ImageDraw
import yaml

names = yaml.safe_load(open(os.path.join(root, "data.yaml")))["names"]
CLASSES = [names[i] for i in sorted(names)]
colors = ["#ff3b30","#34c759","#007aff","#ff9500","#af52de","#00c7be","#ffcc00","#ff2d55","#5856d6","#8e8e93"]

f = random.choice(os.listdir(os.path.join(root, "images/train")))
im = Image.open(os.path.join(root, "images/train", f)).convert("RGB")
d = ImageDraw.Draw(im); W, H = im.size
for line in open(os.path.join(root, "labels/train", f.replace(".png", ".txt"))):
    p = line.split()
    if len(p) != 5: continue
    c = int(p[0]); cx, cy, w, h = map(float, p[1:])
    d.rectangle([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H], outline=colors[c % len(colors)], width=3)
    d.text(((cx-w/2)*W+3, max(0,(cy-h/2)*H-11)), CLASSES[c], fill=colors[c % len(colors)])
print(f, "|", CLASSES)
im


## 3. Train

`epochs` and `imgsz` are the knobs: raise epochs for accuracy, lower imgsz for speed.


In [ ]:
from ultralytics import YOLO

EPOCHS = 60          # 60 is a good start; 100+ if you generated a large dataset
IMGSZ  = 640
MODEL  = "yolo11n.pt"   # nano = fast. yolo11s.pt / yolo11m.pt are stronger and slower.

model = YOLO(MODEL)
results = model.train(
    data=os.path.join(root, "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    patience=15,
    project="megaai",
    name="ui-detector",
    seed=42,
)


## 4. Evaluate on the held-out split


In [ ]:
metrics = model.val()
print("mAP50    :", round(float(metrics.box.map50), 4))
print("mAP50-95 :", round(float(metrics.box.map), 4))
print("precision:", round(float(metrics.box.mp), 4))
print("recall   :", round(float(metrics.box.mr), 4))
print("\nPer class:")
for i, name in enumerate(CLASSES):
    try:
        print(f"  {name:10s} mAP50={float(metrics.box.ap50[i]):.3f}")
    except Exception:
        pass


## 5. See it predict

Runs the trained model on a validation image it never trained on.


In [ ]:
import random
val_dir = os.path.join(root, "images/val")
sample = os.path.join(val_dir, random.choice(os.listdir(val_dir)))
pred = model.predict(sample, conf=0.35, verbose=False)[0]
print(f"{len(pred.boxes)} elements detected in {os.path.basename(sample)}")
Image.fromarray(pred.plot()[:, :, ::-1])


## 6. Export and download

`.onnx` is what the Node engine loads — same weights, no Python needed at inference.


In [ ]:
best = "megaai/ui-detector/weights/best.pt"
model = YOLO(best)
onnx_path = model.export(format="onnx", imgsz=IMGSZ, opset=12)

import shutil
shutil.copy(best, "ui-detector.pt")
shutil.copy(str(onnx_path), "ui-detector.onnx")
print("ready:", os.path.getsize("ui-detector.pt")//1024, "KB .pt |", os.path.getsize("ui-detector.onnx")//1024, "KB .onnx")

try:
    from google.colab import files
    files.download("ui-detector.pt")
    files.download("ui-detector.onnx")
except Exception as e:
    print("Not on Colab — the files are in the working directory.", e)


---
Send `ui-detector.onnx` back and it gets wired into the vision + desktop engines,
so MegaAI can find UI elements from pixels alone.
